# Baselines and Linear Regression: Validation V1

Notebook-led, offline experiment: pinned cleaned snapshot -> new audit -> causal features -> train-only baseline/OLS comparison -> plots -> verification. JSON files are authoritative; cells call tested package APIs. No live collection, random splits, MLP, or final-test scoring.

## 1. Set Up Environment and Paths
Run from this experiment directory or the repository root. Restart the kernel and Run All for a new immutable execution. The current models are deterministic and use CPU; there is no optimizer or random-seed tuning.

In [ ]:
from pathlib import Path
from time import perf_counter
import sys
import pandas as pd
from IPython.display import Image, JSON, display
from oslo_energy.ingestion.archive import Archive
from oslo_energy.pipeline.experiment import ExperimentExecution
from oslo_energy.pipeline.benchmark import validate_benchmark_run

experiment_dir = Path.cwd().resolve()
if not (experiment_dir / 'feature_config.json').is_file():
    experiment_dir = experiment_dir / 'experiments' / '002_baselines_linear_validation_v1'
assert (experiment_dir / 'feature_config.json').is_file(), 'Run from the experiment directory or repository root'
stage_seconds = {}
print('Interpreter:', sys.executable)
print('Experiment:', experiment_dir)

## 2. Load and Snapshot Experiment 002 Settings

The feature config owns inputs, task, windows, and families. The benchmark config owns models and plots. Its original pinned feature run remains available for the historical benchmark-only command; this controller explicitly writes a resolved copy bound to freshly generated features. Source settings are never rewritten.

In [ ]:
execution = ExperimentExecution.begin(experiment_dir)
display(JSON({'experiment_run_dir': str(execution.run_dir)}))
display(JSON(execution.feature_config.effective))
display(JSON(execution.benchmark_config.effective))
print('Source checksums:', execution.source_hashes)

## 3. Confirm the 001 Migration

Experiment 001's source configuration was consolidated into 002 during the repository migration. This notebook performs no cleanup or deletion. Old archives, feature runs, benchmark states, and manifests remain untouched; historical source versions remain in Git.

In [ ]:
assert not (experiment_dir.parent / '001_hour_ahead_v1' / 'config.json').exists(), 'Use the consolidated 002 configuration'
print('Consolidated feature settings:', execution.feature_config.path)
print('No historical artifacts deleted by this execution.')

## 4. Audit Data and Rebuild Causal Features

Recreate the configured chronological windows, audit the pinned cleaned snapshot, then generate exact elapsed-hour lags and known calendar features. No random split, interpolation, target filling, or global fitted preprocessing is used. Pre-window history is available only through issue-time checks. Missing test labels may be described by the audit; test model scores are never computed.

In [ ]:
started = perf_counter()
audit = execution.audit()
stage_seconds['audit'] = perf_counter() - started
display(pd.DataFrame([{'split': item['name'], 'expected_hours': item['expected_hour_count'], 'present_hours': item['present_hour_count'], 'missing_consumption': item['consumption_null_count']} for item in audit.report['windows']]))
started = perf_counter()
features = execution.features()
stage_seconds['features'] = perf_counter() - started
display(pd.DataFrame(features.report['counts_by_split']).T[['intended_examples', 'target_available', 'features_available', 'complete_examples']])
print('Resolved feature config:', execution.resolved_configs['features'])

## 5. Fit the Models and Score Validation

The benchmark adapter fits StandardScaler and ordinary least squares only on eligible training examples. Seasonal naive and persistence remain unfitted. No optimizer, scheduler, epoch search, or MLP is needed for these methods. Model checkpoints are saved by the package, and the benchmark's run-local config explicitly pins this execution's new features.

In [ ]:
started = perf_counter()
benchmark = execution.benchmark()
stage_seconds['benchmark'] = perf_counter() - started
metrics = Archive(benchmark.output_dir / 'metrics.json').read()
comparison = pd.DataFrame([{'model': name, 'MAE': item['errors']['mae'], 'RMSE': item['errors']['rmse'], 'predicted': item['predicted'], 'scored': item['scored'], 'intended': item['intended'], 'coverage': item['prediction_coverage_of_intended']} for name, item in metrics['individual'].items()]).set_index('model')
display(comparison)
print('Resolved benchmark config:', execution.resolved_configs['benchmark'])

## 6. Verify the End-to-End Execution

The preceding cells are the complete audit -> features -> train -> validation sequence. This cell verifies stage integrity, recomputes metrics from saved predictions, and reloads all models to check prediction parity. It deliberately does not evaluate the final test or retrain models.

In [ ]:
started = perf_counter()
checks = execution.verify()
stage_seconds['verification'] = perf_counter() - started
display(JSON(checks))
display(pd.Series(stage_seconds, name='seconds').to_frame())
assert checks['evaluation_split'] == 'validation' and checks['final_test_scored'] is False

## 7. Compare Models and the Original Run

The registered diagram shows shared-timestamp errors, coverage, and paired skill. Errors are in unverified provider units, not percentage accuracy. Compare against the explicitly identified original run when it exists locally; do not select an arbitrary latest run. This is reproduction checking, not tuning.

In [ ]:
for relative_path in benchmark.report['visualizations']:
    display(Image(filename=str(benchmark.output_dir / relative_path)))
display(pd.DataFrame([{'baseline': name, 'paired_count': item['model']['count'], 'OLS_MAE_skill': item['mae_skill']} for name, item in metrics['paired_against_baselines']['linear_regression'].items()]))
original_run = experiment_dir / 'runs' / 'benchmark_78e2f4c89c5549d2a0d06d7a1839c019'
if (original_run / 'manifest.json').is_file():
    validate_benchmark_run(original_run)
    original_metrics = Archive(original_run / 'metrics.json').read()
    display(pd.DataFrame([{'model': name, 'original_MAE': original_metrics['individual'][name]['errors']['mae'], 'rerun_MAE': item['errors']['mae'], 'MAE_delta': item['errors']['mae'] - original_metrics['individual'][name]['errors']['mae']} for name, item in metrics['individual'].items()]))
else:
    print('Original local run unavailable; no historical comparison was fabricated.')

## 8. Publish the Versioned Execution

Write the experiment-level report and checksum manifest only after all checks pass. Config snapshots, resolved bindings, stage outputs, model states, and plots live under this new execution. The clean-kernel runner also saves an executed notebook and a separate notebook completion checksum. No earlier run is overwritten. Source accuracy, actual publication times, and revisions remain unresolved; training readiness is not granted.

In [ ]:
report = execution.complete()
display(JSON(report['checks']))
print('Completed execution:', execution.run_dir)
print('Research eligibility remains pending review; training_ready=false; final test unscored.')